In [1]:
from collections import defaultdict
import re
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from PIL import Image
from torchvision import transforms
from classifier_model_one_out import AIClassifier
import sklearn
import numpy as np

In [6]:
"""
Prereqs: download 10 vids/directory in all_dirs via download_ff_data.sh
"""
def group_by_video(directory):
    groups = defaultdict(list)
    for video_name in os.listdir(directory):
        video_path = os.path.join(directory, video_name)
        if not os.path.isdir(video_path):
            continue
        frames = sorted(f for f in os.listdir(video_path) if f.endswith('.png'))
        if frames:
            # change: doesn't prepend video name to frame filename
            groups[video_name] = [f for f in frames]
    
    result = []
    for key, frames in groups.items():
        result.append([key, frames])
    return result


def extract_original_vid(path: str):
    # use this to ensure that original videos and their manipulated derivatives 
    # are all in the same side of the split, so that the model is only evaluated on data 
    # it hasn't seen before

    name = Path(path).name
    
    # matches DFD fakes, first number is the og, then name should match
    m = re.fullmatch(r'(\d+)_\d+__(.+)__([A-Z0-9]{8})', name)
    if m:
        return f"{m.group(1)}__{m.group(2)}"

    # DFD reals
    m = re.fullmatch(r'(\d+)__(.+)', name)
    if m:
        return f"{m.group(1)}__{m.group(2)}" 

    # rest of the fakes, first number is the target
    m = re.fullmatch(r'(\d+)_(\d+)', name)
    if m:
        return m.group(1)   

    # FF reals
    if re.fullmatch(r'\d+', name):
        return name

    return None 

In [2]:
all_dirs = ['FF_data20/real/original_sequences/youtube/c40/images', 
            'FF_data20/fake/manipulated_sequences/Deepfakes/c40/images',
            'FF_data20/DFD_real/original_sequences/actors/c40/images',
            'FF_data20/DFD_fake/manipulated_sequences/DeepFakeDetection/c40/images',
            'FF_data20/Face2Face/manipulated_sequences/Face2Face/c40/images',
            'FF_data20/FaceSwap/manipulated_sequences/FaceSwap/c40/images',
            'FF_data20/NeuralTextures/manipulated_sequences/NeuralTextures/c40/images'
            ]

'''
0 - Authentic
1 - Deepfake
'''
# same thing as the original version, but combines the root dir and vidpath to
# make ID'ing unique vids easier
# skip is the increment of frames btwn frames in a clip
def create_datamap(all_dirs, frames_per_clip=10, skip=1):
    data = {
        'video_path': [],
        "video_name": []
    }
    for i in range(10):
        data['frame_' + str(i)] = []
    
    data['label'] = []

    for dir in all_dirs:
        label = 1
        if 'original' in dir:
            label = 0
        # list of videos in dir, along with a list of frame filenames with them 
        #[[vid, [vid/fr1.png, vid/fr2.png, ...]], ...]
        video_map = group_by_video(dir)
        for video, frames in video_map:
            i = 0
            for j in range(frames_per_clip * skip, len(frames), frames_per_clip * skip):
                if skip == 1:
                    split = frames[i:j]
                else:
                    split = [frames[idx] for idx in range(i, j, skip)]
                i = j
                data['video_path'].append(os.path.join(dir, video))
                data["video_name"].append(video)
                data['label'].append(label)
                for k in range(len(split)):
                    data['frame_' + str(k)].append(split[k])
    

    return data

source_dir = "dataset_50v_80_20"
remake_data = False
if remake_data:
    data_map = create_datamap(all_dirs)
    df = pd.DataFrame(data_map)
    df['original_vid'] = df['video_name'].apply(extract_original_vid)
    os.makedirs(source_dir, exist_ok=True)

    df.to_csv(os.path.join(source_dir, 'full_dataset.csv'), index=False) # for testing dataset
    vidnames = list(set(df["original_vid"]))
    train_vids, test_vids = train_test_split(vidnames, test_size=0.1, random_state=42)
    train_frames = df.loc[[og_vid in train_vids for og_vid in df["original_vid"]]]
    test_frames = df.loc[[og_vid in test_vids for og_vid in df["original_vid"]]]
    train_frames.to_csv(os.path.join(source_dir, "train_split.csv"), index=False)
    test_frames.to_csv(os.path.join(source_dir, "test_split.csv"), index=False)
else:
    df = pd.read_csv(os.path.join(source_dir, "full_dataset.csv"))
    vidnames = list(set(df["original_vid"]))
    train_frames = pd.read_csv(os.path.join(source_dir, "train_split.csv"))
    test_frames = pd.read_csv(os.path.join(source_dir, "test_split.csv"))
print(f"Num videos: {len(vidnames)}")
print(f"Total clips = {len(df)}")
print(f"Train clips: {len(train_frames)}   Test clips: {len(test_frames)}")

Num videos: 140
Total clips = 820
Train clips: 654   Test clips: 166


In [8]:
if not torch.cuda.is_available():
    print("!!!\n\tCUDA not available. Everything ok?\n!!!")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

class FrameDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.df)
    
    def get_labels(self):
        return self.df["label"]

    def __getitem__(self, idx):
        """
        Returns a clip tensor of shape (3, T, H, W) = (3, 10, 500, 500).
        The DataLoader will stack these into (B, 3, T, H, W).
        """
        vidpath = self.df["video_path"].iloc[idx]
        tensors = []
        for i in range(10):
            fname = self.df[f"frame_{i}"].iloc[idx]  # cols 1–10 are frame_0–frame_9
            path = os.path.join(vidpath, fname)
            img = Image.open(path).convert('RGB')
            tensors.append(self.transform(img)) # (3, 500, 500)
        # TODO normalize pixel vals
        clip = torch.stack(tensors, dim=1).to(dtype=torch.float32) # (3, T, 500, 500)
        og_label = self.df["label"].iloc[idx]
        label = torch.Tensor([og_label])
        return clip, label, torch.Tensor([idx])

train_transform = transforms.Compose([
    transforms.Resize((500, 500)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

test_transform = transforms.Compose([
    transforms.Resize((500, 500)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_dataset = FrameDataset(train_frames, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_dataset = FrameDataset(test_frames, transform=test_transform)


cuda


In [9]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

def test_model(model, data, device='cpu', do_print=False):
    model.to(device=device)
    model.eval()
    total = 0
    correct = 0
    labels = ["Real", "Fake"]
    record = []
    out_text = []
    incorrect = []
    with torch.no_grad():
        for i, (frames, label, idx) in enumerate(data):
            frames = frames.unsqueeze(0).to(device) # add batch dimension
            output = model(frames)[0] # since batch size is 1, we will get only one output
            y_pred = torch.round(nn.functional.sigmoid(output)).item()
            y_actual = label
            record.append((y_pred, y_actual))
            if y_actual == y_pred:
                correct = correct+1
            else:
                incorrect.append(idx)
            total = total+1
    y_pred = [rec[0] for rec in record]
    y_true = [rec[1] for rec in record]
    out_text.append(
        classification_report(
            y_true, 
            y_pred, 
            labels=[0, 1],                              # force both classes
            target_names=['Fake (0)', 'Real (1)'],
            zero_division=0                             # handle missing classes gracefully
        )
    )
    out_text.append("Confusion Matrix:")
    out_text.append(str(confusion_matrix(y_true, y_pred, labels=[0, 1])))
    out_text.append(f"\nF1 Score (weighted): {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    out_text.append(f"F1 Score (macro):    {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    out_text.append(f"Incorrect Indices: {[int(idx.item()) for idx in incorrect]}")
    out_text = "\n".join(out_text)
    if do_print:
        print(out_text)
    return correct / total * 100, out_text

In [13]:
import pickle
from torchvision.ops import sigmoid_focal_loss
from tqdm import tqdm
def train_model(
    model, 
    train_loader, 
    criterion, 
    optimizer, 
    num_epochs, 
    test_data, 
    output_dir, 
    start_epoch=0, 
    losses=[], 
    vals=[],
    accs=[],
    device='cpu', 
    save_every=10
):
    os.makedirs(os.path.join(output_dir, "checkpoints"), exist_ok=True)
    model.to(device)
    best_val = max(vals) if len(vals) > 0 else -1
    for epoch in range(start_epoch, num_epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1} / {num_epochs}")
        for frames, labels, idx in progress_bar:
            # Moving the data to GPU if available
            frames, labels = frames.to(device=device), labels.to(device=device)
            optimizer.zero_grad()
            outputs = model(frames)
            #loss = criterion(outputs, labels)
            loss = sigmoid_focal_loss(outputs, labels, alpha=0.25, gamma=1.0, reduction='mean')
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            preds = torch.round(nn.functional.sigmoid(outputs))
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        accs.append(correct / total)
        losses.append(total_loss / len(train_loader))
        val, output = test_model(model, test_data, device)
        vals.append(val)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}, Train Acc: {100 * correct / total:.2f}, Val Acc: {val:.2f}")

        with open(os.path.join(output_dir, "stats.pkl"), "wb") as f:
            pickle.dump({"epoch": epoch + 1, "loss": losses, "val": vals}, f)
        
        if val > best_val:
            print(f" ! New best val: {val:.4f} !\nSaving to epc_{epoch + 1:04d}_val_{val:.3f}.pkl")
            best_val = val
            torch.save(
                {"epoch": epoch + 1, "losses": losses, "vals": vals, "accs": accs, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "output": output},
                os.path.join(output_dir, "checkpoints", f"epc_{epoch + 1:04d}_val_{val:.3f}.pkl")
            )
        elif (epoch + 1) % save_every == 0 or (epoch + 1) == num_epochs:
            print(f"Checkpoint epoch: saving to epch_{epoch + 1:04d}_val_{val:.3f}.pkl")
            torch.save(
                {"epoch": epoch + 1, "losses": losses, "vals": vals, "accs": accs, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "output": output}, 
                os.path.join(output_dir, "checkpoints", f"epch_{epoch + 1:04d}_val_{val:.3f}.pkl")
            )
        print(output)
    
    return losses, vals, accs

In [14]:
# weight classes to account for imbalance
# weights = sklearn.utils.class_weight.compute_class_weight('balanced', classes=np.unique(train_dataset.get_labels()), y=train_dataset.get_labels())
# numfake = len([1 for lab in train_dataset.get_labels() if lab == 1])
# numreal = len([1 for lab in train_dataset.get_labels() if lab == 0])
# pos_weight = torch.Tensor([numreal / numfake])
# print(pos_weight)
criterion = nn.BCEWithLogitsLoss()
criterion.to(device=device)

BCEWithLogitsLoss()

In [15]:
lr = .0001
epochs = 10
output_dir = "output_50v_final_SFL"

model = AIClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
model.to(device)
# set None for fresh training, file name in the checkpoints directory to load
saved_model = None
if saved_model:
    chkpt = torch.load(os.path.join(output_dir, "checkpoints", saved_model))
    model.load_state_dict(chkpt["model"])
    optimizer.load_state_dict(chkpt["optimizer"])
    start = chkpt["epoch"]
    losses = chkpt["losses"]
    vals = chkpt["vals"]
    accs = chkpt.get("accs", [])
    output = chkpt["output"]
    print(f"Starting from checkpoint {saved_model} at epoch {start}")
    print(output)
else:
    start = 0
    losses = []
    vals = []
    
losses, vals, accs = train_model(
    model, 
    train_loader, 
    criterion, 
    optimizer, 
    epochs, 
    test_dataset, 
    output_dir, 
    start_epoch=start, 
    losses=losses, 
    vals=vals, 
    device=device
)


Epoch 1 / 10: 100%|██████████| 82/82 [04:55<00:00,  3.61s/it]


Epoch 1/10, Loss: 0.1496, Train Acc: 44.95, Val Acc: 27.11
 ! New best val: 27.1084 !
Saving to epc_0001_val_27.108.pkl
              precision    recall  f1-score   support

    Fake (0)       0.27      1.00      0.43        45
    Real (1)       0.00      0.00      0.00       121

    accuracy                           0.27       166
   macro avg       0.14      0.50      0.21       166
weighted avg       0.07      0.27      0.12       166

Confusion Matrix:
[[ 45   0]
 [121   0]]

F1 Score (weighted): 0.1156
F1 Score (macro):    0.2133
Incorrect Indices: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 

Epoch 2 / 10: 100%|██████████| 82/82 [04:58<00:00,  3.64s/it]


Epoch 2/10, Loss: 0.1397, Train Acc: 48.93, Val Acc: 36.75
 ! New best val: 36.7470 !
Saving to epc_0002_val_36.747.pkl
              precision    recall  f1-score   support

    Fake (0)       0.29      0.93      0.44        45
    Real (1)       0.86      0.16      0.27       121

    accuracy                           0.37       166
   macro avg       0.58      0.55      0.36       166
weighted avg       0.71      0.37      0.31       166

Confusion Matrix:
[[ 42   3]
 [102  19]]

F1 Score (weighted): 0.3142
F1 Score (macro):    0.3551
Incorrect Indices: [0, 14, 15, 20, 21, 22, 23, 24, 25, 26, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 65, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 116, 121, 123, 124, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 141, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 157,

Epoch 3 / 10: 100%|██████████| 82/82 [04:56<00:00,  3.62s/it]


...

Epoch 8/10, Loss: 0.1137, Train Acc: 59.48, Val Acc: 39.16
              precision    recall  f1-score   support

    Fake (0)       0.30      0.91      0.45        45
    Real (1)       0.86      0.20      0.32       121

    accuracy                           0.39       166
   macro avg       0.58      0.55      0.39       166
weighted avg       0.71      0.39      0.36       166

Confusion Matrix:
[[41  4]
 [97 24]]

F1 Score (weighted): 0.3563
F1 Score (macro):    0.3851
Incorrect Indices: [0, 14, 15, 19, 20, 21, 22, 23, 24, 25, 26, 29, 30, 31, 34, 35, 36, 37, 38, 39, 65, 66, 67, 68, 69, 72, 73, 74, 75, 76, 77, 78, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 116, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 141, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 157, 160, 161, 162, 163, 164, 165]


Epoch 9 / 10: 100%|██████████| 82/82 [04:53<00:00,  3.58s/it]


Epoch 9/10, Loss: 0.1049, Train Acc: 67.43, Val Acc: 51.81
 ! New best val: 51.8072 !
Saving to epc_0009_val_51.807.pkl
              precision    recall  f1-score   support

    Fake (0)       0.30      0.58      0.39        45
    Real (1)       0.76      0.50      0.60       121

    accuracy                           0.52       166
   macro avg       0.53      0.54      0.50       166
weighted avg       0.63      0.52      0.54       166

Confusion Matrix:
[[26 19]
 [61 60]]

F1 Score (weighted): 0.5441
F1 Score (macro):    0.4970
Incorrect Indices: [0, 4, 5, 6, 7, 10, 11, 14, 15, 19, 22, 23, 24, 25, 29, 30, 31, 34, 35, 36, 37, 44, 45, 46, 47, 48, 49, 56, 57, 58, 67, 69, 76, 77, 78, 83, 84, 86, 87, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 107, 108, 110, 111, 112, 116, 123, 124, 125, 126, 127, 128, 129, 130, 131, 136, 137, 141, 144, 145, 146, 147, 150, 152, 153, 157, 160, 161, 162, 163]


Epoch 10 / 10: 100%|██████████| 82/82 [04:57<00:00,  3.63s/it]


Epoch 10/10, Loss: 0.1063, Train Acc: 69.88, Val Acc: 47.59
Checkpoint epoch: saving to epch_0010_val_47.590.pkl
              precision    recall  f1-score   support

    Fake (0)       0.32      0.80      0.45        45
    Real (1)       0.83      0.36      0.50       121

    accuracy                           0.48       166
   macro avg       0.57      0.58      0.47       166
weighted avg       0.69      0.48      0.49       166

Confusion Matrix:
[[36  9]
 [78 43]]

F1 Score (weighted): 0.4851
F1 Score (macro):    0.4750
Incorrect Indices: [0, 6, 7, 10, 11, 12, 13, 14, 15, 20, 21, 22, 23, 24, 25, 29, 30, 31, 33, 36, 37, 65, 67, 68, 69, 70, 72, 73, 74, 75, 76, 77, 78, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 116, 121, 129, 130, 131, 134, 135, 136, 137, 141, 143, 146, 147, 150, 151, 152, 153, 157, 159, 162, 163]


In [4]:
loaded = torch.load(os.path.join(output_dir, "checkpoints", "epch_0015_val_53.659.pkl"))

FileNotFoundError: [Errno 2] No such file or directory: 'output_50v_final_SFL2/checkpoints/epch_0015_val_53.659.pkl'

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, len(loaded["losses"] + 1)), loaded["losses"], marker='o', color='steelblue')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot(range(1, epochs+1), loaded["accs"], marker='o', color='darkorange')
ax2.set_title('Training Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.grid(True)

plt.suptitle('Our Classifier Model', fontsize=14)
plt.tight_layout()
#plt.savefig('/content/baseline_training_curves.png')
plt.show()